# Solar Filament Segmentation Challenge 2026 -- MVP1 (Kaggle GPU)

Clones the `jp-mvp1` branch of the repo and drives the same `src/train.py` /
`src/infer.py` / `src/submission.py` used locally -- logic lives in one place (the
repo), not duplicated into notebook cells. Local training on a Mac (MPS/CPU) is slow
enough to matter for MVP1 iteration speed; this notebook exists to run the identical
pipeline on a real GPU instead.

MVP1 is not about score -- see `MVP1_PLAN.md` section 0. The goal here is: train the
tiny U-Net faster, produce a validated `submission.csv`, and compare local PQ against
the leaderboard once uploaded.


### 0. Clone the repo

Requires the `jp-mvp1` branch to already be pushed to `origin` -- if this clone
fails with "Remote branch jp-mvp1 not found", push it from local first.


In [ ]:
!git clone -b jp-mvp1 https://github.com/jprakash-1/Solar-Filament-Segmentation.git


In [ ]:
%cd Solar-Filament-Segmentation
!ls


In [ ]:
# re-run-safe: pick up any commits pushed after the kernel started
!git pull origin jp-mvp1


### 1. Install dependencies + confirm GPU

**Do not** `pip install torch torchvision` here -- Kaggle's base image ships a PyTorch build already matched to whatever GPU this kernel was assigned (P100, T4, etc.). Reinstalling from PyPI pulls the newest wheel, which has dropped kernel support for older cards (hit this directly: a CUDA error 'no kernel image is available for execution on the device' on a P100 -- current PyPI PyTorch only ships sm_70+ kernels, P100 is sm_60). Install everything **except** torch/torchvision from requirements.txt; if a genuine version mismatch shows up later, change the notebook's GPU accelerator (Settings -> Accelerator -> T4 x2) rather than reinstalling torch.

In [ ]:
!grep -v -E "^(torch|torchvision)\b" requirements.txt > /tmp/requirements_kaggle.txt
!pip install -q -r /tmp/requirements_kaggle.txt

In [ ]:
import torch
print("torch", torch.__version__)
print("cuda available:", torch.cuda.is_available())
print("gpu count:", torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    print(f"  device {i}:", torch.cuda.get_device_name(i))


### 2. Locate the mounted data

Auto-detects the competition data under `/kaggle/input` by globbing for the
annotation json rather than hardcoding the dataset-mount folder name (which depends
on how the dataset was attached to this kernel) -- falls back to the local `data/raw`
layout so this same cell works when the notebook is run outside Kaggle too.


In [ ]:
!find /kaggle/input -maxdepth 4 2>/dev/null || echo "(not running on Kaggle -- /kaggle/input doesn't exist)"


In [ ]:
import glob
from pathlib import Path

if Path("/kaggle/input").exists():
    matches = glob.glob("/kaggle/input/**/MAGFiLO_1.0_Annotations_kaggle2026_train.json", recursive=True)
    assert matches, "Could not find the training annotation json under /kaggle/input -- check the attached dataset and adjust this cell."
    DATA_JSON = Path(matches[0])
    TRAIN_IMAGES_DIR = DATA_JSON.parent / "train_images"
    TEST_IMAGES_DIR = DATA_JSON.parent.parent / "test" / "test_images"
else:
    DATA_JSON = Path("data/raw/MAGFiLO_1.0_Kaggle_2026/train/MAGFiLO_1.0_Annotations_kaggle2026_train.json")
    TRAIN_IMAGES_DIR = Path("data/raw/MAGFiLO_1.0_Kaggle_2026/train/train_images")
    TEST_IMAGES_DIR = Path("data/raw/MAGFiLO_1.0_Kaggle_2026/test/test_images")

print("DATA_JSON:", DATA_JSON, "exists:", DATA_JSON.exists())
print("TRAIN_IMAGES_DIR:", TRAIN_IMAGES_DIR, "exists:", TRAIN_IMAGES_DIR.exists())
print("TEST_IMAGES_DIR:", TEST_IMAGES_DIR, "exists:", TEST_IMAGES_DIR.exists())
assert DATA_JSON.exists() and TRAIN_IMAGES_DIR.exists() and TEST_IMAGES_DIR.exists(), \
    "adjust the path-detection logic above to match this dataset's actual mount layout"


### 3. Train

GPU affords a bigger batch size and more epochs than the local MPS defaults (see `configs/mvp1_kaggle.yaml` for the reasoning). Launched via `torchrun` so it trains with DistributedDataParallel across every visible GPU (e.g. Kaggle's T4 x2) instead of just the first one -- `--nproc_per_node` auto-detects the GPU count so this cell works unchanged whether Kaggle assigns 1 or 2 GPUs. `--batch-size` is **per GPU** under DDP (32 here x 2 GPUs = 64 effective batch size).

In [ ]:
!torchrun --nproc_per_node=$(python -c "import torch; print(max(1, torch.cuda.device_count()))") -m src.train \
    --data-json "{DATA_JSON}" \
    --images-dir "{TRAIN_IMAGES_DIR}" \
    --img-size 256 \
    --epochs 30 \
    --batch-size 32 \
    --checkpoint-out outputs/checkpoints/mvp1_unet_kaggle.pt \
    --log-csv outputs/logs/train_log_kaggle.csv


### 4. Training curve

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

log = pd.read_csv("outputs/logs/train_log_kaggle.csv")
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(log["epoch"], log["train_loss"], label="train_loss")
axes[0].plot(log["epoch"], log["val_loss"], label="val_loss")
axes[0].set_xlabel("epoch"); axes[0].legend(); axes[0].set_title("Loss")
axes[1].plot(log["epoch"], log["train_dice"], label="train_dice")
axes[1].plot(log["epoch"], log["val_dice"], label="val_dice")
axes[1].set_xlabel("epoch"); axes[1].legend(); axes[1].set_title("Dice")
fig.tight_layout()
plt.show()


### 5. Local validation (Panoptic Quality + Dice)

Runs the full inference + postprocessing pipeline against the held-out
`file_name`-grouped val split -- the same instance-splitting logic used for the real
submission -- as a local sanity check before spending one of the 5 daily submission
slots.


In [ ]:
!python -m src.infer \
    --checkpoint outputs/checkpoints/mvp1_unet_kaggle.pt \
    --split val \
    --data-json "{DATA_JSON}" \
    --train-images-dir "{TRAIN_IMAGES_DIR}"


### 6. Inference -> submission.csv

Predicts on the real test set, splits into filament instances (resize-then-threshold-
then-connected-components, `src/postprocess.py`), RLE-encodes each one, and writes
`filament_id,segmentation_rle` rows.


In [ ]:
!python -m src.infer \
    --checkpoint outputs/checkpoints/mvp1_unet_kaggle.pt \
    --split test \
    --test-images-dir "{TEST_IMAGES_DIR}" \
    --out outputs/submissions/mvp1_unet_kaggle.csv


In [ ]:
!python -m src.submission --validate outputs/submissions/mvp1_unet_kaggle.csv --test-images-dir "{TEST_IMAGES_DIR}"


### 7. Stage for one-click Kaggle submission

Copies the validated CSV to `/kaggle/working/submission.csv` -- Kaggle's convention
for "Submit to competition" directly from a notebook's output.


In [ ]:
import shutil

shutil.copy("outputs/submissions/mvp1_unet_kaggle.csv", "/kaggle/working/submission.csv")
pd.read_csv("/kaggle/working/submission.csv").head()


### Outputs

- `outputs/checkpoints/mvp1_unet_kaggle.pt` -- model weights
- `outputs/logs/train_log_kaggle.csv` -- per-epoch train/val loss + Dice
- `outputs/submissions/mvp1_unet_kaggle.csv` / `/kaggle/working/submission.csv` --
  ready to submit
- Local PQ printed in section 5 -- compare against the leaderboard PQ after
  submitting; a large gap flags a bug in local metric computation or the
  train/val split, not necessarily a bad model (`MVP1_PLAN.md` section 4, day 2 step 8).
